# NAS-Bench-201 — GDE3 & qEHVI (native multi-objective)

Companion to `NAS-mo-pipeline.ipynb` (Study 3). Adds the two **native** multi-objective search methods that were missing from the main pipeline:

- **GDE3** — Generalised Differential Evolution 3 (pymoo)
- **qEHVI** — q-Expected Hypervolume Improvement (BoTorch) — needs `pip install botorch`

Both optimise **f₁ = 1 − test_acc** and **f₂ = normalised FLOPs**, return one Pareto front per seed (no alpha sweep), and are scored by **hypervolume**. Results are written to the same `MO_NAS/<algo>/<dataset>/` folders as the main notebook, so `aggregate_mo_nas.py` picks them up automatically.

## 1. Imports & path setup

In [1]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from datetime import datetime as _dt

ROOT = Path.cwd()

# Point this at the *uncompressed* NAS-Bench-201 directory (same as main notebook).
NAS_PATH = ROOT / 'utils' / 'nas_records' / 'NATS-tss-v1_0' / 'NATS-tss-v1_0'

if (ROOT / 'utils').exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / 'utils').exists():
    sys.path.insert(0, str(ROOT.parent))

from utils.nas import getApi, N_ARCHS, queryArchitecture
from utils.nas_mo import (
    computeFlopsRange,
    computeTrueParetoFront,
    computeHypervolume,
    getNonDominated,
    runGDE3NASSearch,
    runQEHVINASSearch,
    REF_POINT,
)

print('Imports OK')
print('ROOT    :', ROOT)
print('NAS_PATH:', NAS_PATH)

Imports OK
ROOT    : /workspace/code
NAS_PATH: /workspace/code/utils/nas_records/NATS-tss-v1_0/NATS-tss-v1_0


## 2. Configuration

Kept identical to the main notebook so the runs are directly comparable.

In [2]:
DATASETS_USED = ['cifar10', 'cifar100', 'ImageNet16-120']
HP            = '200'

# Search budget — must match Study 2 / the main MO notebook.
N_GENERATIONS = 50
POP_SIZE      = 20
N_QUERIES     = (N_GENERATIONS + 1) * POP_SIZE   # = 1020
SEEDS         = [0, 1, 2]

# GDE3 — native multi-objective Differential Evolution.
GDE3_CR      = 0.9
GDE3_F       = 0.5
GDE3_VARIANT = 'DE/rand/1/bin'

# qEHVI — native multi-objective Bayesian optimisation (needs botorch).
# Total queries = QEHVI_INIT + QEHVI_N_ITER * QEHVI_BATCH. Spend the budget
# through the batch q (=5) rather than 1000 single-point GP refits.
QEHVI_INIT   = POP_SIZE                              # 20
QEHVI_BATCH  = 5
QEHVI_N_ITER = (N_QUERIES - QEHVI_INIT) // QEHVI_BATCH   # 200  -> 20 + 200*5 = 1020
QEHVI_MC     = 128

RESULTS_ROOT = ROOT / 'colab' / 'results'
if not RESULTS_ROOT.exists():
    RESULTS_ROOT = ROOT.parent / 'code' / 'colab' / 'results'
MO_ROOT   = RESULTS_ROOT / 'MO_NAS'
CACHE_DIR = MO_ROOT / 'flops'
MO_ROOT.mkdir(parents=True, exist_ok=True)

print('SEEDS        :', SEEDS)
print('N_QUERIES/run:', N_QUERIES)
print('qEHVI budget :', QEHVI_INIT, '+', QEHVI_N_ITER, 'x', QEHVI_BATCH,
      '=', QEHVI_INIT + QEHVI_N_ITER * QEHVI_BATCH)
print('Datasets     :', DATASETS_USED)
print('Output root  :', MO_ROOT)

SEEDS        : [0, 1, 2]
N_QUERIES/run: 1020
qEHVI budget : 20 + 200 x 5 = 1020
Datasets     : ['cifar10', 'cifar100', 'ImageNet16-120']
Output root  : /workspace/code/colab/results/MO_NAS


## 3. Load NAS-Bench-201 API

In [3]:
nas_path = Path(NAS_PATH)
if not nas_path.exists():
    raise FileNotFoundError(
        f'NAS_PATH does not exist: {nas_path}\n'
        f'  Download NATS-tss-v1_0-3ffb9-simple.tar from '
        f'https://github.com/D-X-Y/NATS-Bench and extract it.')

print('Loading the NAS-Bench-201 API (~30 s first time)...')
api = getApi(file_path=str(nas_path))
print('API loaded. Total architectures:', N_ARCHS)

Loading the NAS-Bench-201 API (~30 s first time)...
API loaded. Total architectures: 15625


## 4. Pre-compute FLOPs range & true Pareto front

Reuses the on-disk cache written by the main notebook if it is present.

In [4]:
flops_ranges = {}
true_fronts  = {}

for dataset in DATASETS_USED:
    print('---', dataset, '---')
    fmin, fmax = computeFlopsRange(api, dataset=dataset, hp=HP, save_dir=str(CACHE_DIR), verbose=True)
    flops_ranges[dataset] = (fmin, fmax)

    tf = computeTrueParetoFront(api, dataset=dataset, hp=HP, save_dir=str(CACHE_DIR), verbose=True)
    true_fronts[dataset] = tf

    with open(MO_ROOT / f'true_front_{dataset}.json', 'w') as f:
        json.dump(tf, f, indent=2)

    tf_hv = computeHypervolume([[p['f1'], p['f2']] for p in tf])
    print(f'  |true front| = {len(tf)}  HV = {tf_hv:.6f}')

--- cifar10 ---
  Loading FLOPs range from /workspace/code/colab/results/MO_NAS/flops/flops_range_cifar10_hp200.json
  Loading true Pareto front from /workspace/code/colab/results/MO_NAS/flops/true_front_cifar10_hp200.json
  |true front| = 17  HV = 1.140140
--- cifar100 ---
  Loading FLOPs range from /workspace/code/colab/results/MO_NAS/flops/flops_range_cifar100_hp200.json
  Loading true Pareto front from /workspace/code/colab/results/MO_NAS/flops/true_front_cifar100_hp200.json
  |true front| = 13  HV = 0.900813
--- ImageNet16-120 ---
  Loading FLOPs range from /workspace/code/colab/results/MO_NAS/flops/flops_range_ImageNet16-120_hp200.json
  Loading true Pareto front from /workspace/code/colab/results/MO_NAS/flops/true_front_ImageNet16-120_hp200.json
  |true front| = 13  HV = 0.610671


## 5. GDE3 — native multi-objective Differential Evolution

One Pareto front per seed (no alpha sweep). Same population budget as the EAs.

In [ ]:
gde3_summary = []

for dataset in DATASETS_USED:
    print('#' * 70)
    print('#  GDE3  --', dataset)
    fmin, fmax = flops_ranges[dataset]
    out_dir = MO_ROOT / 'gde3' / dataset
    out_dir.mkdir(parents=True, exist_ok=True)

    for seed in SEEDS:
        front, run = runGDE3NASSearch(
            api=api, dataset=dataset, hp=HP,
            n_generations=N_GENERATIONS, pop_size=POP_SIZE, seed=seed,
            CR=GDE3_CR, F=GDE3_F, variant=GDE3_VARIANT,
            flops_min=fmin, flops_max=fmax,
            results_dir=str(out_dir), verbose=False)
        tf_hv = computeHypervolume([[p['f1'], p['f2']] for p in true_fronts[dataset]])
        gde3_summary.append({
            'dataset': dataset, 'algorithm': 'GDE3', 'seed': seed,
            'hv': run['hypervolume'], 'hv_gap': tf_hv - run['hypervolume'],
            'front_size': len(front),
            'best_acc': max(p['test_acc'] for p in front),
            'min_flops': min(p['flops_M'] for p in front),
            'n_evals': run['n_evaluations']})

df_gde3 = pd.DataFrame(gde3_summary)
print('\nGDE3 summary:')
print(df_gde3.groupby('dataset')[['hv', 'front_size', 'best_acc']]
      .agg(['mean', 'std']).round(4).to_string())

## 6. qEHVI — native multi-objective Bayesian optimisation

**Slow**: refits a Gaussian process every iteration, so expect minutes per seed. Requires `pip install botorch`. Reduce `QEHVI_N_ITER` for a quick smoke run.

In [7]:
qehvi_summary = []

for dataset in DATASETS_USED:
    print('#' * 70)
    print('#  qEHVI  --', dataset)
    fmin, fmax = flops_ranges[dataset]
    out_dir = MO_ROOT / 'qehvi' / dataset
    out_dir.mkdir(parents=True, exist_ok=True)

    for seed in SEEDS:
        front, run = runQEHVINASSearch(
            api=api, dataset=dataset, hp=HP,
            n_iter=QEHVI_N_ITER, init_points=QEHVI_INIT,
            batch_size=QEHVI_BATCH, mc_samples=QEHVI_MC, seed=seed,
            flops_min=fmin, flops_max=fmax,
            results_dir=str(out_dir), verbose=False)
        tf_hv = computeHypervolume([[p['f1'], p['f2']] for p in true_fronts[dataset]])
        qehvi_summary.append({
            'dataset': dataset, 'algorithm': 'qEHVI', 'seed': seed,
            'hv': run['hypervolume'], 'hv_gap': tf_hv - run['hypervolume'],
            'front_size': len(front),
            'best_acc': max(p['test_acc'] for p in front),
            'min_flops': min(p['flops_M'] for p in front),
            'n_evals': run['n_evaluations']})

df_qehvi = pd.DataFrame(qehvi_summary)
print('\nqEHVI summary:')
print(df_qehvi.groupby('dataset')[['hv', 'front_size', 'best_acc']]
      .agg(['mean', 'std']).round(4).to_string())

######################################################################
#  qEHVI  -- cifar10

  MO-NAS: qEHVI  (cifar10, hp=200)
  Init: 20  |  BO iters: 200 x q=5  |  Total queries: 1020  |  Seed: 0



/usr/local/lib/python3.10/dist-packages/botorch/acquisition/multi_objective/monte_carlo.py:110: NumericsWarning: qExpectedHypervolumeImprovement has known numerical issues that lead to suboptimal optimization performance. It is strongly recommended to simply replace

	 qExpectedHypervolumeImprovement 	 --> 	 qLogExpectedHypervolumeImprovement 

instead, which fixes the issues and has the same API. See https://arxiv.org/abs/2310.20708 for details.
  legacy_ei_numerics_warning(legacy_name=type(self).__name__)
/usr/local/lib/python3.10/dist-packages/botorch/acquisition/multi_objective/monte_carlo.py:110: NumericsWarning: qExpectedHypervolumeImprovement has known numerical issues that lead to suboptimal optimization performance. It is strongly recommended to simply replace

	 qExpectedHypervolumeImprovement 	 --> 	 qLogExpectedHypervolumeImprovement 

instead, which fixes the issues and has the same API. See https://arxiv.org/abs/2310.20708 for details.
  legacy_ei_numerics_warning(legacy_

  iter  20/200  evals= 120  HV=1.134494


/usr/local/lib/python3.10/dist-packages/botorch/acquisition/multi_objective/monte_carlo.py:110: NumericsWarning: qExpectedHypervolumeImprovement has known numerical issues that lead to suboptimal optimization performance. It is strongly recommended to simply replace

	 qExpectedHypervolumeImprovement 	 --> 	 qLogExpectedHypervolumeImprovement 

instead, which fixes the issues and has the same API. See https://arxiv.org/abs/2310.20708 for details.
  legacy_ei_numerics_warning(legacy_name=type(self).__name__)
/usr/local/lib/python3.10/dist-packages/botorch/acquisition/multi_objective/monte_carlo.py:110: NumericsWarning: qExpectedHypervolumeImprovement has known numerical issues that lead to suboptimal optimization performance. It is strongly recommended to simply replace

	 qExpectedHypervolumeImprovement 	 --> 	 qLogExpectedHypervolumeImprovement 

instead, which fixes the issues and has the same API. See https://arxiv.org/abs/2310.20708 for details.
  legacy_ei_numerics_warning(legacy_

  iter  40/200  evals= 220  HV=1.135155


...

  iter 160/200  evals= 820  HV=0.603314


/usr/local/lib/python3.10/dist-packages/botorch/acquisition/multi_objective/monte_carlo.py:110: NumericsWarning: qExpectedHypervolumeImprovement has known numerical issues that lead to suboptimal optimization performance. It is strongly recommended to simply replace

	 qExpectedHypervolumeImprovement 	 --> 	 qLogExpectedHypervolumeImprovement 

instead, which fixes the issues and has the same API. See https://arxiv.org/abs/2310.20708 for details.
  legacy_ei_numerics_warning(legacy_name=type(self).__name__)
/usr/local/lib/python3.10/dist-packages/botorch/acquisition/multi_objective/monte_carlo.py:110: NumericsWarning: qExpectedHypervolumeImprovement has known numerical issues that lead to suboptimal optimization performance. It is strongly recommended to simply replace

	 qExpectedHypervolumeImprovement 	 --> 	 qLogExpectedHypervolumeImprovement 

instead, which fixes the issues and has the same API. See https://arxiv.org/abs/2310.20708 for details.
  legacy_ei_numerics_warning(legacy_

  iter 180/200  evals= 920  HV=0.603340


/usr/local/lib/python3.10/dist-packages/botorch/acquisition/multi_objective/monte_carlo.py:110: NumericsWarning: qExpectedHypervolumeImprovement has known numerical issues that lead to suboptimal optimization performance. It is strongly recommended to simply replace

	 qExpectedHypervolumeImprovement 	 --> 	 qLogExpectedHypervolumeImprovement 

instead, which fixes the issues and has the same API. See https://arxiv.org/abs/2310.20708 for details.
  legacy_ei_numerics_warning(legacy_name=type(self).__name__)
/usr/local/lib/python3.10/dist-packages/botorch/acquisition/multi_objective/monte_carlo.py:110: NumericsWarning: qExpectedHypervolumeImprovement has known numerical issues that lead to suboptimal optimization performance. It is strongly recommended to simply replace

	 qExpectedHypervolumeImprovement 	 --> 	 qLogExpectedHypervolumeImprovement 

instead, which fixes the issues and has the same API. See https://arxiv.org/abs/2310.20708 for details.
  legacy_ei_numerics_warning(legacy_

  iter 200/200  evals=1020  HV=0.603964

  FINAL — qEHVI  (ImageNet16-120)
  |Pareto front|  : 22 architectures
  Hypervolume     : 0.603964
  Best accuracy   : 0.4685
  Min FLOPs       : 2.0 M
  Total evals     : 1020
  Wallclock       : 2301.3s

  Saved: /workspace/code/colab/results/MO_NAS/qehvi/ImageNet16-120/qEHVI_20260629_033708_seed2.json

qEHVI summary:
                    hv         front_size         best_acc        
                  mean     std       mean     std     mean     std
dataset                                                           
ImageNet16-120  0.6035  0.0013    26.3333  8.3865   0.4656  0.0030
cifar10         1.1362  0.0008    14.6667  3.2146   0.9394  0.0009
cifar100        0.8942  0.0028    20.3333  3.0551   0.7259  0.0042


## 7. Summary & quick Pareto plot

In [ ]:
import matplotlib.pyplot as plt

df_native = pd.concat([df_gde3, df_qehvi], ignore_index=True)
print(df_native.groupby(['dataset', 'algorithm'])[['hv', 'best_acc', 'min_flops']]
      .mean().round(4).to_string())
df_native.to_csv(MO_ROOT / 'mo_nas_gde3_qehvi_summary.csv', index=False)

COLORS  = {'GDE3': '#8c564b', 'qEHVI': '#17becf'}
MARKERS = {'GDE3': 'X',       'qEHVI': '*'}
DIRS    = {'GDE3': 'gde3',    'qEHVI': 'qehvi'}

def _front_from_dir(run_dir):
    pts = []
    for jf in sorted(Path(run_dir).glob('*.json')):
        with open(jf) as fh:
            d = json.load(fh)
        if 'pareto_front' in d:
            pts.extend(d['pareto_front'])
    if not pts:
        return []
    F  = np.array([[p['f1'], p['f2']] for p in pts])
    nd = getNonDominated(F)
    return sorted([pts[i] for i in nd], key=lambda p: p['f2'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, dataset in zip(axes, DATASETS_USED):
    tf = sorted(true_fronts[dataset], key=lambda p: p['f2'])
    ax.step([p['f2'] for p in tf], [p['test_acc'] for p in tf],
            where='post', color='black', ls='--', lw=2, label='True front', zorder=10)
    for algo, folder in DIRS.items():
        front = _front_from_dir(MO_ROOT / folder / dataset)
        if not front:
            continue
        ax.scatter([p['f2'] for p in front], [p['test_acc'] for p in front],
                   color=COLORS[algo], marker=MARKERS[algo], s=55, alpha=0.85,
                   label=algo, zorder=5)
    ax.set_xlabel('Normalised FLOPs'); ax.set_ylabel('Test accuracy')
    ax.set_title(dataset); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='lower right')

fig.suptitle('GDE3 vs qEHVI — Pareto fronts on NAS-Bench-201')
plt.tight_layout()
(MO_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
fig.savefig(str(MO_ROOT / 'figures' / 'gde3_qehvi_pareto.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Done.')